In [72]:
!pip install qiskit qiskit-aer --upgrade

In [74]:
from qiskit import QuantumCircuit,transpile
from qiskit_aer import AerSimulator

In [77]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

l = []
sim = AerSimulator()

for _ in range(3):
    qc = QuantumCircuit(3, 3)

    # Superposition
    qc.h([0, 1, 2])

    # Measure
    qc.measure([0, 1, 2], [0, 1, 2])

    # Run
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1).result()

    counts = result.get_counts()
    binary = list(counts.keys())[0][::-1]  # FIX: correct order

    print("Counts:", counts)
    print("Random binary:", binary)

    # Convert to decimal (clean way)
    decimal = int(binary, 2)

    print("Decimal:", decimal)

    l.append(decimal)

# Combine values into OTP
total = sum(l)
otp = total % 10000

print("\nGenerated values:", l)
print("Final 4-digit OTP:", f"{otp:04d}")



Counts: {'010': 1}
Random binary: 010
Decimal: 2
Counts: {'101': 1}
Random binary: 101
Decimal: 5
Counts: {'000': 1}
Random binary: 000
Decimal: 0

Generated values: [2, 5, 0]
Final 4-digit OTP: 0007


In [8]:
import random
from collections import Counter

# Generate 1000 random integers between 1 and 10
numbers = [random.randint(1, 10) for _ in range(1000)]

# Count occurrences
frequency = Counter(numbers)

# Display results in ordered format
print("Number : Count")
for i in range(1, 11):
    print(f"{i} : {frequency[i]}")

Number : Count
1 : 97
2 : 103
3 : 98
4 : 97
5 : 96
6 : 108
7 : 94
8 : 103
9 : 107
10 : 97


In [84]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_aer import AerSimulator

# -----------------------------
# 1. Load Dataset
# -----------------------------
digits = load_digits()
X = digits.data
y = digits.target

print("Dataset shape:", X.shape)

# -----------------------------
# 2. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 3. Feature Scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# 4. PCA (IMPORTANT for Quantum)
# -----------------------------
pca = PCA(n_components=2)   # reduce to 2 features
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# -----------------------------
# 5. Quantum Feature Map
# -----------------------------
feature_map = ZZFeatureMap(feature_dimension=2, reps=2)

# -----------------------------
# 6. Quantum Kernel
# -----------------------------
sim = AerSimulator()

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    backend=sim
)

# -----------------------------
# 7. Compute Kernel Matrices
# -----------------------------
kernel_train = quantum_kernel.evaluate(x_vec=X_train_pca)
kernel_test = quantum_kernel.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

# -----------------------------
# 8. Train SVM (precomputed kernel)
# -----------------------------
model = SVC(kernel='precomputed')
model.fit(kernel_train, y_train)

# -----------------------------
# 9. Prediction
# -----------------------------
y_pred = model.predict(kernel_test)

# -----------------------------
# 10. Evaluation
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\nQuantum SVM Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# -----------------------------
# 11. Visualization
# -----------------------------
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8))
    ax.set_title(f"Pred: {y_pred[i]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

Dataset shape: (1797, 64)


/tmp/ipykernel_11801/3986442742.py:48: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=2, reps=2)


NameError: name 'FidelityQuantumKernel' is not defined

In [85]:
!pip install qiskit qiskit-aer qiskit-machine-learning scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 255.8 kB/s eta 0:00:00


In [90]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_aer import AerSimulator

# -----------------------------
# Load dataset
# -----------------------------



digits = load_digits()
X = digits.data[:2000]   # LIMIT DATA (important for quantum)
y = digits.target[:2000]

# -----------------------------
# Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Scale
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# PCA (reduce features)
# -----------------------------
pca = PCA(n_components=2)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# -----------------------------
# Quantum Feature Map
# -----------------------------
feature_map = ZZFeatureMap(feature_dimension=2, reps=2)

# -----------------------------
# Quantum Kernel
# -----------------------------
sim = AerSimulator()

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

# -----------------------------
# Kernel matrices
# -----------------------------
kernel_train = quantum_kernel.evaluate(x_vec=X_train)
kernel_test = quantum_kernel.evaluate(x_vec=X_test, y_vec=X_train)
qsvm = svc(kernel = quantum_kernel.evaluate)

# -----------------------------
# Train SVM
# -----------------------------
model = SVC(kernel='precomputed')
model.fit(kernel_train, y_train)

# -----------------------------
# Predict
# -----------------------------
qvsm.fit(X_train, y_train)
y_pred = model.predict(kernel_test)
print("y_pred")

print("Accuracy:", accuracy_score(y_test, y_pred))

/tmp/ipykernel_11801/4197801385.py:49: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=2, reps=2)


KeyboardInterrupt: 